In [8]:
# spark.sql("DROP TABLE IF EXISTS bronze_nasdaq_stocks")

from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType
from pyspark.sql.functions import current_timestamp, lit

# 1. Define explicit schema to prevent data type drift on ingestion
bronze_schema = StructType([
    StructField("Date", StringType(), True),
    StructField("Company", StringType(), True),
    StructField("Open", DoubleType(), True),
    StructField("High", DoubleType(), True),
    StructField("Low", DoubleType(), True),
    StructField("Close", DoubleType(), True),
    StructField("Adj_Close", DoubleType(), True),
    StructField("Volume", LongType(), True),
    StructField("Dividends", DoubleType(), True),
    StructField("Stock_Splits", DoubleType(), True)
])

try:
    # 2. Read raw CSV with strict schema
    df_raw = spark.read.format("csv").option("header", "true").schema(bronze_schema).load("Files/NASDAQ_raw/nasdaq100_latest_raw_data.csv")

    # 3. Inject audit metadata columns for lineage
    df_bronze = df_raw.withColumn("_load_timestamp", current_timestamp()).withColumn("_source_file", lit("nasdaq100_latest_raw_data.csv"))

    # 4. Write to Bronze Delta Table
    df_bronze.write.format("delta").mode("overwrite").saveAsTable("bronze_nasdaq_stocks")

    print("Senior Bronze Layer: Successfully ingested with strict schema and lineage metadata.")

except Exception as e:
    print(f"Bronze Ingestion Failed: {str(e)}")
    raise e

StatementMeta(, f003a312-5cf0-40ad-843b-35155d014324, 10, Finished, Available, Finished, False)

Senior Bronze Layer: Successfully ingested with strict schema and lineage metadata.
